*ML-3.1 Способен осуществлять обоснованный и осознанный выбор классических
алгоритмов МО с учителем и выполнять их параметризацию в зависимости от
поставленной задачи и имеющихся данных и с пониманием математической
природы данных, вычислительных ограничений и требований к
интерпретируемости результатов*

# Лабораторная работа №1. Решение задач классификации и регрессии. Множественная регрессия. Построение множественного классификатора.
## Цели и задачи лабораторной работы
**Цель:** Закрепить теоретические знания и получить практические навыки решения задач множественной регрессии и многоклассовой классификации на данных средней сложности.

**Задачи:**
* Освоить автоматическую предобработку данных со смешанными типами признаков (масштабирование, кодирование).
* Изучить стратегии построения множественных классификаторов (One-vs-Rest, Multinomial).
* Научиться оценивать качество моделей и интерпретировать влияние признаков.
* Построить нелинейные модели множественной регрессии с помощью полиномиальных признаков.


# Краткая теория
## 1. Множественная регрессия

Множественная регрессия моделирует зависимость непрерывной целевой переменной $y$ от нескольких признаков:

$$
y = \beta_0 + \beta_1 X_1 + \beta_2 X_2 + \dots + \beta_n X_n + \epsilon
$$

**Особенности применения:**

- Требует масштабирования числовых признаков (StandardScaler).
- Категориальные признаки требуют кодирования (One-Hot Encoding).
- Для учёта нелинейных зависимостей можно использовать полиномиальные признаки:

$$
y = \beta_0 + \beta_1 X_1 + \beta_2 X_2 + \beta_3 X_1^2 + \beta_4 X_1 X_2 + \dots + \epsilon
$$

---

## 2. Множественный классификатор

Базовая логистическая регрессия решает задачу **бинарной классификации**. Для многоклассовых задач (Multi-class) применяются следующие стратегии:

- **One-vs-Rest (OvR):** Для каждого из $K$ классов обучается отдельный бинарный классификатор («класс $i$» против «всех остальных»).

- **One-vs-One (OvO):** Обучается $\frac{K(K-1)}{2}$ классификаторов для каждой пары классов.

- **Multinomial (Softmax):** Единая модель, предсказывающая вероятности всех классов одновременно через функцию softmax:

$$
P(y = k \mid X) = \frac{e^{\beta_k^T X}}{\sum_{j=1}^{K} e^{\beta_j^T X}}
$$

## Описание датасетов

1. Регрессия: Bike Sharing Dataset

Описание: Предсказание количества прокатов велосипедов на основе погодных условий и временных характеристик.

Особенности:

Смешанные типы признаков: числовые (temp, humidity, windspeed) и категориальные (season, weather, weekday)

Временные паттерны: час дня, день недели, месяц

Требует кодирования категориальных признаков и масштабирования числовых
Наличие нелинейных зависимостей (например, влияние температуры)

2. Классификация: Covertype Dataset

Описание: Предсказание типа лесного покрова (7 классов) на основе картографических данных.

Особенности:

Многоклассовая задача с 7 классами (типы деревьев)

54 признака: числовые (elevation, slope, distance) и бинарные (soil type)

Большой объем данных (~581K объектов), требуется subsample для обучения

Требует масштабирования и работы с дисбалансом классов

In [6]:
# Импорт необходимых библиотек
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_openml, fetch_covtype
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, f1_score, classification_report

import warnings
warnings.filterwarnings('ignore')

# Настройка визуализации
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
print("✅ Все библиотеки успешно импортированы!")

✅ Все библиотеки успешно импортированы!


In [8]:
# ==========================================
# ЧАСТЬ 1. ЗАДАЧА РЕГРЕССИИ (Bike Sharing)
# ==========================================
print("Загрузка датасета Bike Sharing...")
# Используем data_id=42712 для гарантированной загрузки
bike = fetch_openml(data_id=42712, as_frame=True, parser='auto')
X_bike_raw, y_bike = bike.data, bike.target.astype(float)

# 1. Умная очистка: удаляем служебные колонки (ID, даты) и целевые, если они попали в X
cols_to_drop = ['instant', 'dteday', 'casual', 'registered', 'cnt', 'count']
X_bike = X_bike_raw.drop(columns=[c for c in cols_to_drop if c in X_bike_raw.columns])

# 2. Автоматическое определение типов признаков (Best Practice!)
num_features = X_bike.select_dtypes(include=['number']).columns.tolist()
cat_features = X_bike.select_dtypes(include=['object', 'category']).columns.tolist()

print(f"✅ Загружено. Размер: {X_bike.shape}")
print(f"Числовые признаки ({len(num_features)}): {num_features}")
print(f"Категориальные признаки ({len(cat_features)}): {cat_features}")

# Разделение на train/test
X_train, X_test, y_train, y_test = train_test_split(X_bike, y_bike, test_size=0.2, random_state=42)

# 3. Создание пайплайна предобработки
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_features)
    ])

# Базовая модель множественной регрессии
baseline_reg = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

# Обучение и оценка
baseline_reg.fit(X_train, y_train)
y_pred = baseline_reg.predict(X_test)

print(f"\n=== Baseline: Множественная линейная регрессия ===")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.2f}")
print(f"R2: {r2_score(y_test, y_pred):.4f}")

# Анализ коэффициентов
coef_names = (preprocessor.named_transformers_['cat'].get_feature_names_out(cat_features).tolist() +
              num_features)
coef_values = baseline_reg.named_steps['regressor'].coef_

# Топ-10 наиболее важных признаков
coef_df = pd.DataFrame({'feature': coef_names, 'coef_abs': np.abs(coef_values)})
print(f"\nТоп-10 наиболее важных признаков:")
print(coef_df.sort_values('coef_abs', ascending=False).head(10))

Загрузка датасета Bike Sharing...
✅ Загружено. Размер: (17379, 12)
Числовые признаки (8): ['year', 'month', 'hour', 'weekday', 'temp', 'feel_temp', 'humidity', 'windspeed']
Категориальные признаки (4): ['season', 'holiday', 'workingday', 'weather']

=== Baseline: Множественная линейная регрессия ===
RMSE: 137.92
R2: 0.3993

Топ-10 наиболее важных признаков:
               feature   coef_abs
2        season_summer  51.839481
4        holiday_False  50.328003
11        weather_rain  44.300665
0          season_fall  40.667567
6     workingday_False  37.062539
17           feel_temp  36.252519
19           windspeed  32.156382
8        weather_clear  26.017000
9   weather_heavy_rain  19.168509
5         holiday_True  17.863349


#Задание 1. Множественная регрессия с нелинейными признаками
1. Модифицируйте пайплайн, добавив полиномиальные признаки (степень 2) для числовых переменных. Используйте PolynomialFeatures(degree=2, include_bias=False, interaction_only=False).
2. Обучите модель на расширенном наборе признаков и сравните метрики RMSE и
$R^2$ с базовой линейной моделью.
3. Постройте график: зависимость предсказанного количества прокатов от часа дня (признак hr или hour). Используйте scatter plot с прозрачностью.
4. Проанализируйте, какие полиномиальные признаки (квадратичные или взаимодействия) имеют наибольшие коэффициенты.

In [ ]:
# === РАБОЧЕЕ МЕСТО СТУДЕНТА (Задание 1) ===
# Ваш код здесь:
# 1. Создайте пайплайн с PolynomialFeatures для числовых признаков
#    Подсказка: используйте вложенный ColumnTransformer или добавьте шаг в пайплайн
# 2. Обучите модель и сравните метрики с baseline_reg
# 3. Постройте scatter plot: y_pred vs hr (или hour)
# 4. Найдите топ-5 полиномиальных признаков по абсолютному значению коэффициента

In [9]:
# ==========================================
# ЧАСТЬ 2. ЗАДАЧА КЛАССИФИКАЦИИ (Covertype)
# ==========================================
print("Загрузка датасета Covertype...")
# Используем встроенную функцию sklearn для 100% стабильности
covertype = fetch_covtype(as_frame=True)
X_cov_raw, y_cov = covertype.data, covertype.target.astype(int)

print(f"✅ Загружено. Полный размер: {X_cov_raw.shape}")
print(f"\nРаспределение классов:")
print(y_cov.value_counts().sort_index())

# Для ускорения обучения на обычных ноутбуках берем случайную подвыборку (100 000 объектов)
np.random.seed(42)
sample_indices = np.random.choice(len(X_cov_raw), size=100000, replace=False)
X_cov = X_cov_raw.iloc[sample_indices]
y_cov = y_cov.iloc[sample_indices]

# Разделение на train/test (стратифицированное)
X_train_cov, X_test_cov, y_train_cov, y_test_cov = train_test_split(
    X_cov, y_cov, test_size=0.2, random_state=42, stratify=y_cov
)

# Масштабирование (все признаки в Covertype числовые, но имеют разный масштаб)
scaler_cov = StandardScaler()
X_train_cov_scaled = scaler_cov.fit_transform(X_train_cov)
X_test_cov_scaled = scaler_cov.transform(X_test_cov)

# Базовый множественный классификатор: One-vs-Rest
clf_ovr = LogisticRegression(multi_class='ovr', max_iter=1000, random_state=42)
clf_ovr.fit(X_train_cov_scaled, y_train_cov)
y_pred_ovr = clf_ovr.predict(X_test_cov_scaled)

print("\n=== Baseline: One-vs-Rest ===")
print(f"Accuracy: {accuracy_score(y_test_cov, y_pred_ovr):.4f}")
print(f"Weighted F1-score: {f1_score(y_test_cov, y_pred_ovr, average='weighted'):.4f}")
print("\nClassification Report:\n", classification_report(y_test_cov, y_pred_ovr))

# Анализ коэффициентов для класса 1
coef_class_1 = clf_ovr.coef_[0]
top_features_class_1 = pd.DataFrame({
    'feature': X_cov.columns,
    'coef': coef_class_1
}).sort_values('coef', key=abs, ascending=False).head(10)
print(f"\nТоп-10 признаков для класса 1 (по абсолютному значению коэффициента):")
print(top_features_class_1)

Загрузка датасета Covertype...
✅ Загружено. Полный размер: (581012, 54)

Распределение классов:
Cover_Type
1    211840
2    283301
3     35754
4      2747
5      9493
6     17367
7     20510
Name: count, dtype: int64

=== Baseline: One-vs-Rest ===
Accuracy: 0.7128
Weighted F1-score: 0.6961

Classification Report:
               precision    recall  f1-score   support

           1       0.71      0.69      0.70      7333
           2       0.74      0.79      0.76      9732
           3       0.61      0.88      0.72      1229
           4       0.52      0.13      0.21        89
           5       0.00      0.00      0.00       339
           6       0.40      0.05      0.09       589
           7       0.69      0.49      0.58       689

    accuracy                           0.71     20000
   macro avg       0.52      0.43      0.44     20000
weighted avg       0.69      0.71      0.70     20000


Топ-10 признаков для класса 1 (по абсолютному значению коэффициента):
                

# Задание 2. Построение множественного классификатора
1. Обучите модель логистической регрессии с использованием стратегии Multinomial (multi_class='multinomial').
2. Сравните метрики Accuracy и Weighted F1-score между стратегиями ovr и multinomial. Какая стратегия работает лучше?
3. Проанализируйте classification_report. Какие классы (от 1 до 7) модель предсказывает хуже всего (наименьший F1-score)? Совпадает ли это с их количеством в обучающей выборке?
4. Постройте матрицу ошибок (confusion matrix) для модели ovr с помощью seaborn.heatmap. Какие классы чаще всего путаются?

In [ ]:
# === РАБОЧЕЕ МЕСТО СТУДЕНТА (Задание 2) ===
# Ваш код здесь:
# 1. Обучите clf_multinomial = LogisticRegression(multi_class='multinomial', max_iter=1000)
# 2. Сравните метрики с clf_ovr
# 3. Выведите classification_report для multinomial и сравните с ovr
# 4. Постройте confusion matrix для ovr:
#    from sklearn.metrics import confusion_matrix
#    cm = confusion_matrix(y_test_cov, y_pred_ovr)
#    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')

# Задание 3. Индивидуальное (по вариантам)
Преподаватель назначает вариант по списку в журнале. Выполните дополнительное задание вашего варианта и приложите его в конец ноутбука.

| Вариант | Индивидуальное задание (Регрессия) | Индивидуальное задание (Классификация) |
|---------|-----------------------------------|---------------------------------------|
| **1, 6, 11** | Добавить взаимодействие признаков (`interaction_only=True`) для числовых переменных. Сравнить с полной полиномиальной моделью. | Использовать стратегию **One-vs-One** (`multi_class='ovo'`). Сравнить с OvR и Multinomial. |
| **2, 7, 12** | Добавить полиномиальные признаки **степени 3** для числовых переменных. Сравнить метрики с degree=2. | Использовать только **первые 10 признаков** (elevation, aspect, slope и т.д.). Оценить, как это влияет на качество и скорость. |
| **3, 8, 13** | Добавить **циклическое кодирование** (sin/cos) для признака часа (`hr`). Подсказка: `np.sin(2 * np.pi * hr / 24)`. | Построить **barplot** абсолютных значений коэффициентов для класса 2 (модель ovr). Какие признаки наиболее важны? |
| **4, 9, 14** | Применить **Log-transform** к целевой переменной: `y_log = np.log1p(y)`. Обучить модель на `y_log`, затем сделать `np.expm1(predictions)`. | Использовать только **бинарные признаки** (soil type). Сравнить качество с полной моделью. |
| **5, 10, 15** | Построить график **зависимости предсказаний от температуры** (`temp`) с цветовым кодированием по сезону. | Построить **pairplot** для первых 5 признаков с цветовым кодированием по классам (используйте subsample 5000 объектов). |